# LeetCode 1005 — Maximize Sum Of Array After K Negations bằng Python

**Link đề:** https://leetcode.com/problems/maximize-sum-of-array-after-k-negations/
**Độ khó:** Easy · **Chủ đề:** Array, Greedy (tham lam), Sorting

```python
class Solution:
    def largestSumAfterKNegations(self, nums: list[int], k: int) -> int:
        ...
```

## Mục tiêu học
1. Đọc đúng đề: phải đổi dấu **đúng `k` lần**, được phép đổi cùng một vị trí nhiều lần.
2. Ôn các công cụ Python: `sorted`, `list.sort`, `abs`, `min`, `sum`, `%`, `heapq`, `sorted(key=...)`.
3. Viết lời giải trực giác (thử hết / mô phỏng từng lần đổi dấu) để hiểu bài.
4. Rút ra **insight tham lam**: lật số âm có trị tuyệt đối lớn trước; phần lượt thừa chỉ quan trọng chẵn hay lẻ.
5. Trace từng bước, chứng minh vì sao tham lam đúng, phân tích Time/Space.
6. So sánh 4 cách giải, làm bài tập biến thể.

**Cách học:** đọc markdown → đoán output của cell code kế tiếp → chạy → so sánh. Các cell TODO chỉ để luyện, không tự chạy.

## 1. Đề bài — đổi dấu đúng `k` lần để tổng lớn nhất

Cho mảng số nguyên `nums` và số nguyên `k`. Một **thao tác** là: chọn một chỉ số `i` rồi thay `nums[i]` bằng `-nums[i]`.

- Phải thực hiện thao tác **đúng `k` lần** (không ít hơn, không nhiều hơn).
- Được chọn **cùng một `i` nhiều lần**.

Trả về **tổng lớn nhất có thể** của mảng sau `k` thao tác.

| `nums` | `k` | Output | Giải thích |
|---|:---:|:---:|---|
| `[4,2,3]` | 1 | `5` | đổi dấu số 2 → `[4,-2,3]`, tổng 5 |
| `[3,-1,0,2]` | 3 | `6` | đổi `-1` thành `1`, rồi đổi số `0` hai lần → `[3,1,0,2]` |
| `[2,-3,-1,5,-4]` | 2 | `13` | đổi `-3` và `-4` → `[2,3,-1,5,4]` |

**Constraints:** `1 <= nums.length <= 10^4`, `-100 <= nums[i] <= 100`, `1 <= k <= 10^4`.

Câu hỏi tự đặt trước khi code:
- Đổi dấu một số **âm** làm tổng tăng hay giảm? (Tăng: `-3` → `3` làm tổng tăng 6.)
- Đổi dấu một số **dương** thì sao? (Giảm.) Vậy khi buộc phải đổi số dương, nên chọn số nào?
- Đổi dấu cùng một số 2 lần thì mảng thế nào? (Như cũ — hai lần "triệt tiêu" nhau.)

In [1]:
# Kiểm tra lại ví dụ 3 bằng tay
nums = [2, -3, -1, 5, -4]
print("Tổng ban đầu:", sum(nums))
nums[1] = -nums[1]   # đổi dấu -3
nums[4] = -nums[4]   # đổi dấu -4
print("Sau 2 lần đổi:", nums, "-> tổng", sum(nums))

Tổng ban đầu: -1
Sau 2 lần đổi: [2, 3, -1, 5, 4] -> tổng 13


### 1.1. Hiệu ứng của một lần đổi dấu lên tổng

Nếu đổi `x` thành `-x` thì tổng thay đổi một lượng `(-x) - x = -2x`.

- `x = -3` → tổng thay đổi `+6` (tốt).
- `x = 0` → tổng thay đổi `0` (vô hại!).
- `x = 2` → tổng thay đổi `-4` (xấu).

Vậy: **muốn tổng tăng nhiều nhất, hãy đổi dấu số nhỏ nhất (âm nhất)**. Đây là hạt giống của lời giải tham lam.

In [2]:
for x in [-3, -1, 0, 2, 5]:
    print(f"đổi dấu x = {x:>2} -> tổng thay đổi {-2 * x:+d}")

đổi dấu x = -3 -> tổng thay đổi +6
đổi dấu x = -1 -> tổng thay đổi +2
đổi dấu x =  0 -> tổng thay đổi +0
đổi dấu x =  2 -> tổng thay đổi -4
đổi dấu x =  5 -> tổng thay đổi -10


## 2. Nền tảng Python cần dùng

Bài này cần các "viên gạch" sau:
1. `sum`, `min`, `abs` — tổng, nhỏ nhất, trị tuyệt đối.
2. Sắp xếp: `sorted(nums)` (tạo list mới) và `nums.sort()` (sắp tại chỗ).
3. Sắp xếp theo khóa: `sorted(nums, key=abs, reverse=True)`.
4. Toán tử `%` để biết số chẵn/lẻ: `k % 2`.
5. `heapq` — hàng đợi ưu tiên (min-heap), lấy phần tử nhỏ nhất nhanh.

### 2.1. `sum`, `min`, `abs`

In [3]:
nums = [3, -1, 0, 2]
print("sum  :", sum(nums))         # 3 + (-1) + 0 + 2 = 4
print("min  :", min(nums))         # -1
print("abs  :", abs(-7), abs(7))   # 7 7
print("min theo trị tuyệt đối:", min(abs(x) for x in nums))   # 0

sum  : 4
min  : -1
abs  : 7 7
min theo trị tuyệt đối: 0


Dòng cuối dùng **generator expression** `abs(x) for x in nums`: đọc là "với mỗi `x` trong `nums`, lấy `abs(x)`". `min(...)` nhận lần lượt các giá trị đó và trả giá trị nhỏ nhất — không cần tạo list trung gian.

### 2.2. `sorted(nums)` và `nums.sort()`

- `sorted(nums)` **trả về list mới** đã sắp tăng dần, `nums` giữ nguyên.
- `nums.sort()` **sắp xếp tại chỗ**, trả về `None`.

In [4]:
nums = [2, -3, -1, 5, -4]
a = sorted(nums)
print("sorted :", a, "| nums vẫn là:", nums)
r = nums.sort()
print("sau .sort():", nums, "| giá trị trả về:", r)

sorted : [-4, -3, -1, 2, 5] | nums vẫn là: [2, -3, -1, 5, -4]
sau .sort(): [-4, -3, -1, 2, 5] | giá trị trả về: None


### 2.3. Sắp xếp theo khóa `key=`

`sorted(nums, key=abs)` so sánh các phần tử theo `abs(x)` thay vì theo `x`. Thêm `reverse=True` để sắp **giảm dần**.

Ví dụ `[2,-3,-1,5,-4]` theo trị tuyệt đối giảm dần: `5, -4, -3, 2, -1`.

In [5]:
nums = [2, -3, -1, 5, -4]
print(sorted(nums, key=abs))                 # tăng theo |x|
print(sorted(nums, key=abs, reverse=True))   # giảm theo |x|

[-1, 2, -3, -4, 5]
[5, -4, -3, 2, -1]


### 2.4. `k % 2` — chẵn hay lẻ

`k % 2` là phần dư khi chia `k` cho 2: bằng `0` nếu `k` chẵn, `1` nếu lẻ. Trong bài này: đổi dấu cùng một số **chẵn lần** thì như không đổi; **lẻ lần** thì như đổi 1 lần.

In [6]:
x = 7
for times in range(5):
    y = x
    for _ in range(times):
        y = -y
    print(f"đổi dấu {times} lần: {y:>2}   (times % 2 = {times % 2})")

đổi dấu 0 lần:  7   (times % 2 = 0)
đổi dấu 1 lần: -7   (times % 2 = 1)
đổi dấu 2 lần:  7   (times % 2 = 0)
đổi dấu 3 lần: -7   (times % 2 = 1)
đổi dấu 4 lần:  7   (times % 2 = 0)


### 2.5. `heapq` — min-heap

`heapq` biến một list thành **heap**: phần tử nhỏ nhất luôn ở `heap[0]`.
- `heapq.heapify(h)` — biến list thành heap, O(n).
- `heapq.heappop(h)` — lấy ra phần tử nhỏ nhất, O(log n).
- `heapq.heappush(h, x)` — thêm phần tử, O(log n).
- `heapq.heapreplace(h, x)` — pop nhỏ nhất rồi push `x` trong một bước.

In [7]:
import heapq
h = [4, 2, 3, -1]
heapq.heapify(h)
print("heap:", h, "| nhỏ nhất:", h[0])
smallest = heapq.heappop(h)
print("pop ra:", smallest, "| heap còn:", h)
heapq.heappush(h, 1)        # đẩy -(-1) = 1 vào lại
print("sau push 1:", h, "| nhỏ nhất:", h[0])

heap: [-1, 2, 3, 4] | nhỏ nhất: -1
pop ra: -1 | heap còn: [2, 4, 3]
sau push 1: [1, 2, 3, 4] | nhỏ nhất: 1


## 3. Cách trực giác / Baseline

### 3.1. Cách 0 — Thử hết mọi khả năng (chỉ để kiểm chứng)

Mỗi lần đổi dấu có `n` lựa chọn, `k` lần → `n^k` chuỗi lựa chọn. Quá nhiều, nhưng với mảng nhỏ (`n ≤ 4`, `k ≤ 4`) ta dùng nó làm **"trọng tài" (oracle)** để kiểm tra các cách nhanh.

Ta dùng `itertools.product(range(n), repeat=k)`: sinh mọi bộ `k` chỉ số, mỗi chỉ số trong `0..n-1`.

In [8]:
from itertools import product

print(list(product(range(2), repeat=3)))   # 2^3 = 8 bộ

def largest_sum_bruteforce(nums, k):
    n = len(nums)
    best = None
    for choice in product(range(n), repeat=k):
        arr = nums[:]               # bản sao để không phá nums gốc
        for i in choice:
            arr[i] = -arr[i]
        s = sum(arr)
        if best is None or s > best:
            best = s
    return best

print(largest_sum_bruteforce([4, 2, 3], 1))        # 5
print(largest_sum_bruteforce([3, -1, 0, 2], 3))    # 6

[(0, 0, 0), (0, 0, 1), (0, 1, 0), (0, 1, 1), (1, 0, 0), (1, 0, 1), (1, 1, 0), (1, 1, 1)]
5
6


`nums[:]` là **slicing** lấy toàn bộ list → tạo một **bản sao nông**. Nếu viết `arr = nums` thì `arr` và `nums` cùng trỏ một list, sửa `arr` là hỏng `nums`.

### 3.2. Cách 1 — Mô phỏng tham lam: mỗi lần đổi dấu số nhỏ nhất

Từ mục 1.1: một lần đổi dấu làm tổng thay đổi `-2x` → chọn `x` nhỏ nhất là tốt nhất **cho lần đó**. Cứ lặp `k` lần:
1. Tìm chỉ số của phần tử nhỏ nhất.
2. Đổi dấu nó.

In [9]:
def largest_sum_simulate(nums, k):
    arr = nums[:]
    for _ in range(k):
        i = arr.index(min(arr))   # vị trí phần tử nhỏ nhất
        arr[i] = -arr[i]
    return sum(arr)

print(largest_sum_simulate([4, 2, 3], 1))           # 5
print(largest_sum_simulate([3, -1, 0, 2], 3))       # 6
print(largest_sum_simulate([2, -3, -1, 5, -4], 2))  # 13

5
6
13


`arr.index(v)` trả về **chỉ số đầu tiên** có giá trị `v`. `min(arr)` duyệt n phần tử, `index` thêm n nữa → mỗi vòng O(n), tổng **O(k·n)**.

## 4. Nút thắt: vì sao cần cải tiến?

Với `n = 10^4`, `k = 10^4`: `k·n = 10^8` phép so sánh — quá chậm với Python.

Quan sát hai việc lặp đi lặp lại vô ích:
1. **Tìm min lặp lại từ đầu** mỗi vòng → dùng heap (O(log n)) hoặc sắp xếp một lần.
2. Khi mọi số đã **không âm**, các lần đổi tiếp theo chỉ "đá qua đá lại" trên **cùng một số nhỏ nhất**. Không cần mô phỏng từng lần — chỉ cần biết số lượt còn lại **chẵn hay lẻ**.

In [10]:
# Quan sát nút thắt 2: mảng toàn số dương, k lớn
def trace_simulate(nums, k):
    arr = nums[:]
    for step in range(1, k + 1):
        i = arr.index(min(arr))
        arr[i] = -arr[i]
        print(f"lần {step}: đổi dấu vị trí {i} -> {arr}, tổng = {sum(arr)}")

trace_simulate([4, 2, 3], 5)

lần 1: đổi dấu vị trí 1 -> [4, -2, 3], tổng = 5
lần 2: đổi dấu vị trí 1 -> [4, 2, 3], tổng = 9
lần 3: đổi dấu vị trí 1 -> [4, -2, 3], tổng = 5
lần 4: đổi dấu vị trí 1 -> [4, 2, 3], tổng = 9
lần 5: đổi dấu vị trí 1 -> [4, -2, 3], tổng = 5


Thấy rõ: sau lần 1, các lần còn lại chỉ bật tắt số `2` ↔ `-2`. Kết quả cuối chỉ phụ thuộc **còn lại 4 lần (chẵn)** → như không đổi.

## 5. Insight — tham lam theo hai pha

**Pha 1 — biến âm thành dương:** sắp xếp tăng dần; đi từ trái sang, gặp số âm và còn lượt thì đổi dấu. Số âm "lớn nhất về trị tuyệt đối" đứng đầu → được đổi trước, tăng tổng nhiều nhất.

**Pha 2 — xử lý lượt thừa:** nếu số lượt còn lại `k` là **lẻ**, buộc phải đổi dấu một số (không âm) một lần → chọn số có **trị tuyệt đối nhỏ nhất**, tổng giảm `2 * min_abs`. Nếu `k` chẵn: đổi đi đổi lại cùng một số → không mất gì.

Lưu ý: nếu mảng có số `0`, `min_abs = 0` → lượt lẻ không làm mất gì.

Công thức: `đáp án = sum(mảng sau pha 1) - (2 * min_abs nếu k còn lẻ, ngược lại 0)`.

In [11]:
# Tính các giá trị trung gian cho ví dụ [3,-1,0,2], k=3
nums = sorted([3, -1, 0, 2])
k = 3
print("sau sort:", nums)
for i in range(len(nums)):
    if nums[i] < 0 and k > 0:
        nums[i] = -nums[i]
        k -= 1
print("sau pha 1:", nums, "| k còn:", k)
min_abs = min(nums)          # sau pha 1 mọi số đều >= 0 HOẶC còn số âm nhưng k đã = 0
print("min_abs:", min_abs, "| k lẻ?", k % 2 == 1)
print("đáp án:", sum(nums) - (2 * min_abs if k % 2 == 1 else 0))

sau sort: [-1, 0, 2, 3]
sau pha 1: [1, 0, 2, 3] | k còn: 2
min_abs: 0 | k lẻ? False
đáp án: 6


### 5.1. Một chi tiết tinh tế: `min_abs` lấy sau pha 1

Sau pha 1, nếu `k` vẫn > 0 thì **mọi số âm đã bị đổi** → mảng toàn số ≥ 0, nên `min(nums)` chính là trị tuyệt đối nhỏ nhất.

Nếu `k` đã về 0 trong pha 1 thì pha 2 không làm gì (0 là chẵn) → không cần `min_abs`.

Ví dụ khó: `[-4,-2,1]`, `k=3`. Sau sort: `[-4,-2,1]`. Pha 1 đổi `-4`, `-2` → `[4,2,1]`, `k=1`. Lẻ → trừ `2*1`. Đáp án `7 - 2 = 5`. Chú ý số bị đổi lần cuối là `1` (số dương gốc), **không phải** `2` (số vừa đổi từ âm).

Còn `[-5,-1,3]`, `k=3`: pha 1 → `[5,1,3]`, `k=1` → trừ `2*1` → `9 - 2 = 7`. Ở đây số nhỏ nhất lại là số vừa đổi từ `-1`. Vì vậy phải lấy min **trên toàn mảng sau pha 1**, không được chỉ nhìn số dương gốc.

In [12]:
def two_phase(nums, k):
    nums = sorted(nums)
    for i in range(len(nums)):
        if nums[i] < 0 and k > 0:
            nums[i] = -nums[i]
            k -= 1
    if k % 2 == 1:
        return sum(nums) - 2 * min(nums)
    return sum(nums)

print(two_phase([-4, -2, 1], 3), "== bruteforce", largest_sum_bruteforce([-4, -2, 1], 3))
print(two_phase([-5, -1, 3], 3), "== bruteforce", largest_sum_bruteforce([-5, -1, 3], 3))

5 == bruteforce 5
7 == bruteforce 7


## 6. Lời giải chính — mổ xẻ từng đoạn rồi ghép lại

### 6.1. Đoạn 1: sắp xếp

```python
nums.sort()
```
Đưa các số âm "âm nhất" lên đầu. Ở LeetCode được phép sửa `nums`; trong notebook ta sắp trên bản sao để không phá dữ liệu test.

### 6.2. Đoạn 2: vòng lặp pha 1

```python
for i in range(len(nums)):
    if nums[i] < 0 and k > 0:
        nums[i] = -nums[i]
        k -= 1
```
- `range(len(nums))` sinh chỉ số `0..n-1` vì ta cần **ghi** vào `nums[i]`.
- Điều kiện `and k > 0`: hết lượt thì dừng đổi.
- Có thể `break` sớm khi gặp số ≥ 0 (vì đã sắp xếp, phía sau không còn số âm).

### 6.3. Đoạn 3: pha 2 và trả kết quả

```python
total = sum(nums)
if k % 2 == 1:
    total -= 2 * min(nums)
return total
```

In [13]:
def largest_sum_greedy_sort(nums: list[int], k: int) -> int:
    nums = sorted(nums)                # Đoạn 1 (bản sao)
    for i in range(len(nums)):         # Đoạn 2
        if nums[i] >= 0 or k == 0:
            break
        nums[i] = -nums[i]
        k -= 1
    total = sum(nums)                  # Đoạn 3
    if k % 2 == 1:
        total -= 2 * min(nums)
    return total

for nums, k in [([4, 2, 3], 1), ([3, -1, 0, 2], 3), ([2, -3, -1, 5, -4], 2)]:
    print(nums, k, "->", largest_sum_greedy_sort(nums, k))

[4, 2, 3] 1 -> 5
[3, -1, 0, 2] 3 -> 6
[2, -3, -1, 5, -4] 2 -> 13


Kết quả `5, 6, 13` khớp ba ví dụ của đề.

Vì sao `break` an toàn? Mảng đã tăng dần; khi `nums[i] >= 0` thì mọi phần tử sau cũng `>= 0`, không còn số âm nào để đổi.

## 7. TRACE từng bước — nhìn thấy trạng thái biến

Ví dụ `nums = [2,-3,-1,5,-4]`, `k = 4` (nhiều lượt hơn số âm để thấy pha 2).

Sau sort: `[-4,-3,-1,2,5]`.

| i | nums[i] trước | k trước | Hành động | nums sau | k sau |
|:-:|:-:|:-:|---|---|:-:|
| 0 | -4 | 4 | âm, còn lượt → đổi | `[4,-3,-1,2,5]` | 3 |
| 1 | -3 | 3 | âm, còn lượt → đổi | `[4,3,-1,2,5]` | 2 |
| 2 | -1 | 2 | âm, còn lượt → đổi | `[4,3,1,2,5]` | 1 |
| 3 | 2 | 1 | không âm → `break` | — | 1 |

Pha 2: tổng = 15, `k = 1` lẻ, `min = 1` → `15 - 2 = 13`.

Ý nghĩa: lượt thừa rơi vào số `1` (gốc là `-1`), đúng như mục 5.1 cảnh báo.

In [14]:
def trace_greedy_sort(nums, k):
    arr = sorted(nums)
    print("Sau sort:", arr, "| k =", k)
    for i in range(len(arr)):
        if arr[i] >= 0 or k == 0:
            print(f"  i={i}: arr[i]={arr[i]}, k={k} -> dừng pha 1")
            break
        before = arr[i]
        arr[i] = -arr[i]
        k -= 1
        print(f"  i={i}: đổi {before} -> {arr[i]}, arr={arr}, k còn {k}")
    total = sum(arr)
    print("Tổng sau pha 1:", total)
    if k % 2 == 1:
        m = min(arr)
        print(f"k={k} lẻ -> trừ 2*{m}")
        total -= 2 * m
    else:
        print(f"k={k} chẵn -> giữ nguyên")
    print("Kết quả:", total)
    return total

trace_greedy_sort([2, -3, -1, 5, -4], 4)

Sau sort: [-4, -3, -1, 2, 5] | k = 4
  i=0: đổi -4 -> 4, arr=[4, -3, -1, 2, 5], k còn 3
  i=1: đổi -3 -> 3, arr=[4, 3, -1, 2, 5], k còn 2
  i=2: đổi -1 -> 1, arr=[4, 3, 1, 2, 5], k còn 1
  i=3: arr[i]=2, k=1 -> dừng pha 1
Tổng sau pha 1: 15
k=1 lẻ -> trừ 2*1
Kết quả: 13


13

### 7.1. Trace một ca có số 0

`[3,-1,0,2]`, `k=3`: số `0` hấp thụ mọi lượt thừa.

In [15]:
trace_greedy_sort([3, -1, 0, 2], 3)
print()
trace_greedy_sort([-8, -5, -5, -1], 2)   # k hết trước khi đổi hết số âm

Sau sort: [-1, 0, 2, 3] | k = 3
  i=0: đổi -1 -> 1, arr=[1, 0, 2, 3], k còn 2
  i=1: arr[i]=0, k=2 -> dừng pha 1
Tổng sau pha 1: 6
k=2 chẵn -> giữ nguyên
Kết quả: 6

Sau sort: [-8, -5, -5, -1] | k = 2
  i=0: đổi -8 -> 8, arr=[8, -5, -5, -1], k còn 1
  i=1: đổi -5 -> 5, arr=[8, 5, -5, -1], k còn 0
  i=2: arr[i]=-5, k=0 -> dừng pha 1
Tổng sau pha 1: 7
k=0 chẵn -> giữ nguyên
Kết quả: 7


7

Ca thứ hai: `k = 2` chỉ đủ đổi `-8` và `-5`; `-5`, `-1` còn lại vẫn âm. Vòng lặp dừng do `k == 0`, pha 2 bỏ qua vì `0` chẵn → `8 + 5 - 5 - 1 = 7`.

## 8. Code nộp LeetCode (Python 3) — cách sắp xếp

In [16]:
class Solution:
    def largestSumAfterKNegations(self, nums: list[int], k: int) -> int:
        nums.sort()
        for i in range(len(nums)):
            if nums[i] >= 0 or k == 0:
                break
            nums[i] = -nums[i]
            k -= 1
        total = sum(nums)
        if k % 2 == 1:
            total -= 2 * min(nums)
        return total

sol = Solution()
assert sol.largestSumAfterKNegations([4, 2, 3], 1) == 5
assert sol.largestSumAfterKNegations([3, -1, 0, 2], 3) == 6
assert sol.largestSumAfterKNegations([2, -3, -1, 5, -4], 2) == 13
print("Solution (sort) PASS 3 ví dụ của đề")

Solution (sort) PASS 3 ví dụ của đề


## 9. Các lời giải khác — từ cơ bản tới gọn nhất

### 9.1. Cách 2 — Heap: mô phỏng nhưng lấy min nhanh

Giữ nguyên ý "mỗi lần đổi dấu số nhỏ nhất" của Cách 1, nhưng dùng min-heap để lấy min trong O(log n). Thêm mẹo: nếu số nhỏ nhất đã ≥ 0 thì xử lý chẵn/lẻ và dừng sớm.

In [17]:
import heapq

def largest_sum_heap(nums: list[int], k: int) -> int:
    h = nums[:]
    heapq.heapify(h)
    while k > 0:
        if h[0] >= 0:                 # hết số âm: chỉ còn chẵn/lẻ
            if k % 2 == 1:
                heapq.heapreplace(h, -h[0])
            break
        heapq.heapreplace(h, -h[0])   # đổi dấu số nhỏ nhất
        k -= 1
    return sum(h)

for nums, k in [([4, 2, 3], 1), ([3, -1, 0, 2], 3), ([2, -3, -1, 5, -4], 2), ([-4, -2, 1], 3)]:
    print(nums, k, "->", largest_sum_heap(nums, k))

[4, 2, 3] 1 -> 5
[3, -1, 0, 2] 3 -> 6
[2, -3, -1, 5, -4] 2 -> 13
[-4, -2, 1] 3 -> 5


Code nộp LeetCode (heap). Thời gian **O(n + min(k, n) · log n)**, bộ nhớ O(n) cho heap (O(1) thêm nếu heapify ngay trên `nums`).

In [18]:
class SolutionHeap:
    def largestSumAfterKNegations(self, nums: list[int], k: int) -> int:
        heapq.heapify(nums)
        while k > 0:
            if nums[0] >= 0:
                if k % 2 == 1:
                    heapq.heapreplace(nums, -nums[0])
                break
            heapq.heapreplace(nums, -nums[0])
            k -= 1
        return sum(nums)

assert SolutionHeap().largestSumAfterKNegations([2, -3, -1, 5, -4], 2) == 13
print("SolutionHeap PASS")

SolutionHeap PASS


### 9.2. Cách 3 — Sắp xếp theo trị tuyệt đối giảm dần (lời giải "kinh điển")

Sắp xếp theo `abs` giảm dần. Duyệt từ đầu: gặp số âm và còn lượt thì đổi dấu. Cuối cùng, nếu còn lượt lẻ thì đổi dấu **phần tử cuối** (trị tuyệt đối nhỏ nhất).

Vì sao gọn? Phần tử cuối luôn có |x| nhỏ nhất — không cần gọi `min` nữa, và nó tự xử lý trường hợp "số nhỏ nhất là số vừa đổi từ âm".

In [19]:
def largest_sum_abs_sort(nums: list[int], k: int) -> int:
    arr = sorted(nums, key=abs, reverse=True)
    for i in range(len(arr)):
        if arr[i] < 0 and k > 0:
            arr[i] = -arr[i]
            k -= 1
    if k % 2 == 1:
        arr[-1] = -arr[-1]
    return sum(arr)

for nums, k in [([4, 2, 3], 1), ([3, -1, 0, 2], 3), ([2, -3, -1, 5, -4], 2), ([-5, -1, 3], 3)]:
    print(nums, k, "->", largest_sum_abs_sort(nums, k))

[4, 2, 3] 1 -> 5
[3, -1, 0, 2] 3 -> 6
[2, -3, -1, 5, -4] 2 -> 13
[-5, -1, 3] 3 -> 7


In [20]:
class SolutionAbs:
    def largestSumAfterKNegations(self, nums: list[int], k: int) -> int:
        nums.sort(key=abs, reverse=True)
        for i in range(len(nums)):
            if nums[i] < 0 and k > 0:
                nums[i] = -nums[i]
                k -= 1
        if k % 2 == 1:
            nums[-1] = -nums[-1]
        return sum(nums)

assert SolutionAbs().largestSumAfterKNegations([3, -1, 0, 2], 3) == 6
print("SolutionAbs PASS")

SolutionAbs PASS


### 9.3. Cách 4 — Đếm (counting) vì giá trị nằm trong `[-100, 100]`

Constraints nói `-100 <= nums[i] <= 100` → chỉ có 201 giá trị khác nhau. Ta đếm số lần xuất hiện của mỗi giá trị rồi xử lý từ `-100` lên, **không cần sắp xếp** → O(n + 201).

- `count[v + 100]` là số lần giá trị `v` xuất hiện (cộng 100 để chỉ số không âm).
- Với `v` âm từ `-100` đến `-1`: đổi `t = min(count, k)` bản sao của `v` thành `-v`.

In [21]:
def largest_sum_counting(nums: list[int], k: int) -> int:
    count = [0] * 201
    for x in nums:
        count[x + 100] += 1
    for v in range(-100, 0):
        if k == 0:
            break
        t = min(count[v + 100], k)
        count[v + 100] -= t
        count[-v + 100] += t
        k -= t
    total = sum((idx - 100) * c for idx, c in enumerate(count))
    if k % 2 == 1:
        # tìm giá trị không âm nhỏ nhất còn tồn tại (sau pha 1 nếu k>0 thì không còn số âm)
        for v in range(0, 101):
            if count[v + 100] > 0:
                total -= 2 * v
                break
    return total

for nums, k in [([4, 2, 3], 1), ([3, -1, 0, 2], 3), ([2, -3, -1, 5, -4], 2), ([-4, -2, 1], 3)]:
    print(nums, k, "->", largest_sum_counting(nums, k))

[4, 2, 3] 1 -> 5
[3, -1, 0, 2] 3 -> 6
[2, -3, -1, 5, -4] 2 -> 13
[-4, -2, 1] 3 -> 5


`enumerate(count)` cho từng cặp `(idx, c)`: chỉ số và giá trị. `idx - 100` đổi chỉ số về giá trị thật.

Cách này nhanh nhất về lý thuyết nhưng **phụ thuộc constraints** — nếu giá trị lên tới `10^9` thì không dùng được.

## 10. Vì sao thuật toán đúng? — lập luận ngắn

Gọi `neg` là số lượng số âm.

**Bổ đề 1 (thứ tự đổi):** Nếu chỉ được đổi `t ≤ neg` số âm, tổng lớn nhất khi đổi `t` số âm có |x| lớn nhất. Vì đổi `x` tăng tổng `-2x = 2|x|`; chọn `t` số có |x| lớn nhất cho tổng tăng lớn nhất. Sắp xếp tăng dần đưa đúng các số đó lên đầu.

**Bổ đề 2 (đổi 2 lần = không đổi):** Đổi dấu cùng một vị trí 2 lần trả mảng về như cũ. Vì vậy chỉ **tính chẵn lẻ** của số lần mỗi vị trí bị đổi mới quan trọng.

**Pha 2:** Khi mọi số đã ≥ 0 và còn `k` lượt:
- `k` chẵn: dồn hết vào một vị trí → không mất gì; không thể làm tốt hơn vì đổi bất kỳ số ≥ 0 nào cũng không làm tổng tăng.
- `k` lẻ: tổng số lần đổi lẻ ⇒ ít nhất một vị trí bị đổi lẻ lần ⇒ ít nhất một số đổi dấu ⇒ mất ít nhất `2·min_abs`. Đạt được đúng mức đó bằng cách dồn mọi lượt vào phần tử nhỏ nhất.

**Bất biến của vòng lặp pha 1:** sau khi xử lý chỉ số `i`, các phần tử `nums[0..i]` đã được đổi dấu đúng nếu chúng âm và còn lượt; tổng hiện tại là lớn nhất có thể với số lượt đã dùng.

In [22]:
# Kiểm chứng Bổ đề 2: đổi dấu một vị trí 2 lần = không đổi
arr = [3, -1, 0, 2]
copy = arr[:]
copy[1] = -copy[1]
copy[1] = -copy[1]
print(arr == copy)

True


## 11. Phân tích Time / Space Complexity

Gọi `n = len(nums)`.

| Cách | Ý tưởng | Thời gian | Bộ nhớ thêm |
|---|---|---|---|
| 0. Thử hết | mọi chuỗi `k` lựa chọn | `O(n^k · n)` | `O(n)` |
| 1. Mô phỏng | lặp `k` lần tìm min | `O(k · n)` | `O(n)` (bản sao) |
| 2. Heap | min-heap + dừng sớm | `O(n + min(k,n) log n)` | `O(n)` hoặc `O(1)` tại chỗ |
| 3. Sort tăng / sort theo abs | sắp xếp một lần, hai pha | `O(n log n)` | `O(1)` tại chỗ (Timsort cần tới `O(n)` phụ) |
| 4. Counting | đếm 201 giá trị | `O(n + R)`, `R = 201` | `O(R)` |

Nhận xét:
- Cách 3 dễ viết, dễ chứng minh nhất → nên nộp.
- Cách 4 là tối ưu tiệm cận nhưng gắn chặt với miền giá trị nhỏ.

## 12. Kiểm thử: mẫu đề, biên, phản ví dụ, đối chiếu

Nhóm test:
1. Ba ví dụ của đề.
2. Biên: mảng 1 phần tử, `k` rất lớn chẵn/lẻ.
3. Có số `0` hấp thụ lượt thừa.
4. `k` nhỏ hơn số lượng số âm.
5. Ca bẫy: số nhỏ nhất sau pha 1 là số vừa đổi từ âm (`[-5,-1,3]`, `k=3`).

In [23]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<38} PASS ({actual})")

cases = [
    ("đề 1",                           [4, 2, 3], 1, 5),
    ("đề 2",                           [3, -1, 0, 2], 3, 6),
    ("đề 3",                           [2, -3, -1, 5, -4], 2, 13),
    ("1 phần tử dương, k lẻ",          [5], 1, -5),
    ("1 phần tử dương, k chẵn",        [5], 10000, 5),
    ("1 phần tử âm, k = 2",            [-5], 2, -5),
    ("có số 0",                        [-2, 0, 9], 5, 11),
    ("k < số số âm",                   [-8, -5, -5, -1], 2, 7),
    ("bẫy: min là số vừa đổi",         [-5, -1, 3], 3, 7),
    ("bẫy: min là số dương gốc",       [-4, -2, 1], 3, 5),
    ("toàn số âm, k = n",              [-1, -2, -3], 3, 6),
    ("toàn số âm, k = n + 1",          [-1, -2, -3], 4, 4),
]
for name, nums, k, exp in cases:
    check(name, Solution().largestSumAfterKNegations(nums[:], k), exp)

đề 1                                   PASS (5)
đề 2                                   PASS (6)
đề 3                                   PASS (13)
1 phần tử dương, k lẻ                  PASS (-5)
1 phần tử dương, k chẵn                PASS (5)
1 phần tử âm, k = 2                    PASS (-5)
có số 0                                PASS (11)
k < số số âm                           PASS (7)
bẫy: min là số vừa đổi                 PASS (7)
bẫy: min là số dương gốc               PASS (5)
toàn số âm, k = n                      PASS (6)
toàn số âm, k = n + 1                  PASS (4)


### 12.1. Đối chiếu tất cả các cách với oracle thử hết (random có seed)

`random.seed(1005)` cố định chuỗi số ngẫu nhiên → chạy lại luôn ra cùng kết quả, dễ tái hiện lỗi.

In [24]:
import random
random.seed(1005)

solvers = {
    "simulate": largest_sum_simulate,
    "greedy_sort": largest_sum_greedy_sort,
    "heap": largest_sum_heap,
    "abs_sort": largest_sum_abs_sort,
    "counting": largest_sum_counting,
    "Solution": lambda a, k: Solution().largestSumAfterKNegations(a[:], k),
}
tests = 0
for _ in range(400):
    n = random.randint(1, 4)
    nums = [random.randint(-6, 6) for _ in range(n)]
    k = random.randint(1, 4)
    expected = largest_sum_bruteforce(nums, k)
    for name, f in solvers.items():
        got = f(nums[:], k)
        assert got == expected, (name, nums, k, got, expected)
    tests += 1
print(f"{tests} bộ ngẫu nhiên: mọi cách đều khớp oracle thử hết")

400 bộ ngẫu nhiên: mọi cách đều khớp oracle thử hết


### 12.2. Đối chiếu trên input lớn (không dùng oracle chậm)

Với `n = 10^4`, `k = 10^4`, ta so sánh các cách nhanh với nhau.

In [25]:
random.seed(7)
big = [random.randint(-100, 100) for _ in range(10_000)]
for k in [1, 2, 5000, 9999, 10000]:
    results = {name: f(big[:], k) for name, f in solvers.items() if name != "simulate"}
    assert len(set(results.values())) == 1, results
    print(f"k = {k:>5}: mọi cách cho {results['Solution']}")

k =     1: mọi cách cho -10398
k =     2: mọi cách cho -10198
k =  5000: mọi cách cho 503034


k =  9999: mọi cách cho 503264


k = 10000: mọi cách cho 503264


## 13. Sai lầm người mới hay mắc — code sai, vì sao sai, cách sửa

### 13.1. Lỗi: lấy `min` trên **mảng gốc** thay vì mảng sau pha 1

In [26]:
def wrong_min_original(nums, k):
    original_min_abs = min(x for x in nums if x >= 0) if any(x >= 0 for x in nums) else 0
    arr = sorted(nums)
    for i in range(len(arr)):
        if arr[i] < 0 and k > 0:
            arr[i] = -arr[i]
            k -= 1
    if k % 2 == 1:
        return sum(arr) - 2 * original_min_abs   # SAI: bỏ qua số vừa đổi từ âm
    return sum(arr)

print("sai :", wrong_min_original([-5, -1, 3], 3))
print("đúng:", largest_sum_greedy_sort([-5, -1, 3], 3))

sai : 3
đúng: 7


**Vì sao sai:** sau pha 1, `-1` thành `1` — nhỏ hơn `3`. Lượt thừa nên rơi vào `1` (mất 2) chứ không phải `3` (mất 6). **Sửa:** lấy `min(arr)` sau pha 1, hoặc dùng cách sort theo `abs` và đổi `arr[-1]`.

### 13.2. Lỗi: quên rằng `k` có thể lớn hơn số lượng số âm (dùng hết lượt rồi bỏ qua)

In [27]:
def wrong_ignore_leftover(nums, k):
    arr = sorted(nums)
    for i in range(len(arr)):
        if arr[i] < 0 and k > 0:
            arr[i] = -arr[i]
            k -= 1
    return sum(arr)          # SAI: quên k còn lẻ

print("sai :", wrong_ignore_leftover([4, 2, 3], 1))
print("đúng:", largest_sum_greedy_sort([4, 2, 3], 1))

sai : 9
đúng: 5


**Vì sao sai:** đề bắt đổi **đúng** `k` lần. `[4,2,3]`, `k=1` không có số âm nhưng vẫn phải đổi một số → đáp án 5, không phải 9. **Sửa:** thêm pha 2 với `k % 2`.

### 13.3. Lỗi: trừ `min` thay vì `2 * min`

In [28]:
def wrong_minus_once(nums, k):
    arr = sorted(nums)
    for i in range(len(arr)):
        if arr[i] < 0 and k > 0:
            arr[i] = -arr[i]
            k -= 1
    return sum(arr) - (min(arr) if k % 2 else 0)   # SAI: chỉ trừ 1 lần

print("sai :", wrong_minus_once([4, 2, 3], 1))
print("đúng:", largest_sum_greedy_sort([4, 2, 3], 1))

sai : 7
đúng: 5


**Vì sao sai:** đổi `2` thành `-2` làm tổng giảm từ `+2` xuống `-2`, tức giảm **4 = 2·2**. **Sửa:** `total -= 2 * min(arr)`.

### 13.4. Lỗi: đổi dấu số nhỏ nhất **theo thứ tự xuất hiện** thay vì sau khi sắp xếp

Duyệt `nums` gốc và đổi các số âm gặp đầu tiên: `[-1, -5]`, `k=1` sẽ đổi `-1` → tổng `-4`, trong khi đổi `-5` cho tổng `4`. **Sửa:** sắp xếp trước (hoặc dùng heap).

In [29]:
def wrong_no_sort(nums, k):
    arr = nums[:]
    for i in range(len(arr)):
        if arr[i] < 0 and k > 0:
            arr[i] = -arr[i]
            k -= 1
    return sum(arr) - (2 * min(arr) if k % 2 else 0)

print("sai :", wrong_no_sort([-1, -5], 1))
print("đúng:", largest_sum_greedy_sort([-1, -5], 1))

sai : -4
đúng: 4


## 14. Nhận diện pattern

Dấu hiệu nên nghĩ tới **tham lam + sắp xếp** như bài này:
- "Thực hiện thao tác đúng/tối đa `k` lần để tổng lớn nhất/nhỏ nhất".
- Mỗi thao tác có **lợi ích độc lập** tính được ngay (`-2x`) → chọn thao tác lợi nhất trước.
- Thao tác **tự nghịch đảo** (làm 2 lần = không làm) → chỉ quan tâm **chẵn/lẻ** của phần dư.

Bài liên quan:
- LeetCode 1913 (Maximum Product Difference), 561 (Array Partition) — sort rồi tham lam.
- LeetCode 1046 (Last Stone Weight), 2558 (Take Gifts From the Richest Pile) — heap mô phỏng "lấy lớn/nhỏ nhất lặp lại".
- LeetCode 2530 (Maximal Score After Applying K Operations) — heap + k thao tác.

## 15. Biến thể

### 15.1. Biến thể A — "tối đa `k` lần" thay vì "đúng `k` lần"

Nếu được dừng sớm, pha 2 không còn cần thiết: chỉ đổi các số âm (tối đa `k` số âm nhỏ nhất).

In [30]:
def largest_sum_at_most_k(nums, k):
    arr = sorted(nums)
    for i in range(len(arr)):
        if arr[i] >= 0 or k == 0:
            break
        arr[i] = -arr[i]
        k -= 1
    return sum(arr)

print(largest_sum_at_most_k([4, 2, 3], 1))        # 9: không cần đổi gì
print(largest_sum_at_most_k([-5, -1, 3], 3))      # 9

9
9


### 15.2. Biến thể B — tổng **nhỏ nhất** sau đúng `k` lần đổi

Đối xứng: đổi số **dương lớn nhất** trước. Mẹo: `min_sum(nums, k) = -max_sum([-x for x in nums], k)`.

In [31]:
def smallest_sum_after_k(nums, k):
    return -largest_sum_greedy_sort([-x for x in nums], k)

print(smallest_sum_after_k([4, 2, 3], 1))   # đổi 4 -> -4: -4+2+3 = 1
print(smallest_sum_after_k([-1, 0], 3))     # 0 hấp thụ: đổi... -> -1

1
-1


## 16. Hệ thống luyện tập — 5 bài từ dễ đến nâng cao

### Bài 1 — Tính tay
Cho `nums = [-2, 9, 9, 8, 4]`, `k = 5`. Hãy tự lập bảng trace pha 1 và pha 2, ghi ra đáp án cuối. (Không cần code.)

### Bài 2 — Dự đoán output
Đoán output của `largest_sum_abs_sort([1, -2, -3], 4)` trước khi chạy. Sau pha 1 mảng là gì, phần tử cuối là gì?

### Bài 3 — TODO: tự viết lại lời giải hai pha

In [32]:
def exercise_largest_sum(nums: list[int], k: int) -> int:
    # TODO 1: sắp xếp bản sao của nums tăng dần
    # TODO 2: pha 1 — đổi dấu số âm khi còn lượt
    # TODO 3: pha 2 — nếu k lẻ thì trừ 2 * min
    pass

# Tự làm xong rồi mới mở test:
# assert exercise_largest_sum([4, 2, 3], 1) == 5
# assert exercise_largest_sum([3, -1, 0, 2], 3) == 6
# assert exercise_largest_sum([-5, -1, 3], 3) == 7

### Bài 4 — TODO: trả về **mảng** sau `k` lần đổi (không chỉ tổng)

Trả về một mảng đạt tổng lớn nhất (theo thứ tự sau khi sắp xếp tăng dần của mảng gốc là được).

In [33]:
def exercise_array_after_k(nums: list[int], k: int) -> list[int]:
    # TODO: giống bài 3 nhưng thay vì trừ 2*min, hãy đổi dấu thật phần tử nhỏ nhất
    pass

# assert sum(exercise_array_after_k([4, 2, 3], 1)) == 5
# assert sorted(exercise_array_after_k([3, -1, 0, 2], 3)) == [0, 1, 2, 3]

### Bài 5 — TODO biến thể: mỗi phần tử chỉ được đổi dấu **tối đa một lần**, phải dùng **đúng** `k` lần (`k <= n`)

Giờ không thể "đá qua đá lại" một số. Ví dụ `[4,2,3]`, `k=2` → phải đổi 2 số khác nhau: đổi `2` và `3` → `4-2-3 = -1`.

In [34]:
def exercise_each_once(nums: list[int], k: int) -> int:
    # TODO: chọn đúng k phần tử khác nhau để đổi dấu, tổng lớn nhất
    pass

# assert exercise_each_once([4, 2, 3], 2) == -1
# assert exercise_each_once([-3, -2, 5], 2) == 10
# assert exercise_each_once([-3, -2, 5], 3) == 0

### Bài 6 — Mini challenge: counting sort cho miền giá trị tùy ý

Viết `exercise_counting(nums, k, lo, hi)` dùng kỹ thuật đếm cho mọi `lo <= nums[i] <= hi` (không cố định -100..100).

In [35]:
def exercise_counting(nums: list[int], k: int, lo: int, hi: int) -> int:
    # TODO: mảng đếm kích thước hi - lo + 1, xử lý giá trị âm từ nhỏ tới lớn
    pass

# assert exercise_counting([2, -3, -1, 5, -4], 2, -5, 5) == 13
# assert exercise_counting([1000, -1000], 3, -1000, 1000) == 2000

## 17. Gợi ý ba mức — chỉ mở sau khi tự thử

**Bài 1**
- Mức 1: sort trước.
- Mức 2: chỉ có một số âm; sau khi đổi nó, `k` còn 4.
- Mức 3: 4 là chẵn → không trừ gì.

**Bài 2**
- Mức 1: sort theo `abs` giảm dần: `[-3, -2, 1]`.
- Mức 2: đổi `-3`, `-2`, còn `k = 2`.
- Mức 3: `k` chẵn → không đổi `arr[-1]`.

**Bài 3**
- Mức 1: hai pha như mục 5.
- Mức 2: `for i in range(len(arr))`, điều kiện `arr[i] < 0 and k > 0`.
- Mức 3: `return sum(arr) - (2 * min(arr) if k % 2 else 0)`.

**Bài 4**
- Mức 1: ở pha 2 tìm chỉ số của min.
- Mức 2: `j = arr.index(min(arr))`.
- Mức 3: `if k % 2: arr[j] = -arr[j]`.

**Bài 5**
- Mức 1: mỗi số chỉ bị đổi ≤ 1 lần → chọn tập `k` phần tử.
- Mức 2: đổi `x` làm tổng thay đổi `-2x` → chọn `k` phần tử **nhỏ nhất**.
- Mức 3: `sum(nums) - 2 * sum(sorted(nums)[:k])`.

**Bài 6**
- Mức 1: chỉ số `x - lo`.
- Mức 2: duyệt `v` từ `lo` tới `-1` (nếu `lo < 0`), chuyển đếm sang `-v` (cần `-v <= hi`; nếu không thì mở rộng mảng hoặc dùng `dict`).
- Mức 3: dùng `collections.Counter` cho gọn.

## 18. Đáp án tham khảo — đặt cuối để tránh lộ trước khi làm TODO

**Bài 1:** sort `[-2,4,8,9,9]` → đổi `-2` (k=4) → `[2,4,8,9,9]`, tổng 32, `k=4` chẵn → **32**.

**Bài 2:** `[-3,-2,1]` → `[3,2,1]`, `k=2` chẵn → **6**.

In [36]:
from collections import Counter

def answer_largest_sum(nums, k):
    arr = sorted(nums)
    for i in range(len(arr)):
        if arr[i] < 0 and k > 0:
            arr[i] = -arr[i]
            k -= 1
    return sum(arr) - (2 * min(arr) if k % 2 else 0)


def answer_array_after_k(nums, k):
    arr = sorted(nums)
    for i in range(len(arr)):
        if arr[i] < 0 and k > 0:
            arr[i] = -arr[i]
            k -= 1
    if k % 2 == 1:
        j = arr.index(min(arr))
        arr[j] = -arr[j]
    return arr


def answer_each_once(nums, k):
    return sum(nums) - 2 * sum(sorted(nums)[:k])


def answer_counting(nums, k, lo, hi):
    cnt = Counter(nums)
    for v in range(lo, 0):
        if k == 0:
            break
        t = min(cnt[v], k)
        cnt[v] -= t
        cnt[-v] += t
        k -= t
    total = sum(v * c for v, c in cnt.items())
    if k % 2 == 1:
        smallest = min(v for v, c in cnt.items() if c > 0)
        total -= 2 * smallest
    return total


assert answer_largest_sum([-2, 9, 9, 8, 4], 5) == 32
assert largest_sum_abs_sort([1, -2, -3], 4) == 6
assert answer_largest_sum([4, 2, 3], 1) == 5
assert answer_largest_sum([3, -1, 0, 2], 3) == 6
assert answer_largest_sum([-5, -1, 3], 3) == 7
assert sum(answer_array_after_k([4, 2, 3], 1)) == 5
assert sorted(answer_array_after_k([3, -1, 0, 2], 3)) == [0, 1, 2, 3]
assert answer_each_once([4, 2, 3], 2) == -1
assert answer_each_once([-3, -2, 5], 2) == 10
assert answer_each_once([-3, -2, 5], 3) == 0
assert answer_counting([2, -3, -1, 5, -4], 2, -5, 5) == 13
assert answer_counting([1000, -1000], 3, -1000, 1000) == 2000

random.seed(2024)
for _ in range(300):
    nums = [random.randint(-6, 6) for _ in range(random.randint(1, 4))]
    k = random.randint(1, 4)
    assert answer_counting(nums, k, -6, 6) == largest_sum_bruteforce(nums, k)
    assert sum(answer_array_after_k(nums, k)) == largest_sum_bruteforce(nums, k)
print("Đáp án tham khảo PASS toàn bộ test!")

Đáp án tham khảo PASS toàn bộ test!


**Giải thích bài 5:** mỗi số chỉ đổi ≤ 1 lần nên chọn đúng `k` phần tử khác nhau. Đổi `x` làm tổng thay đổi `-2x` → để tổng lớn nhất, chọn `k` phần tử **nhỏ nhất** (kể cả khi chúng dương — vì bắt buộc đúng `k` lần).

**Bài 6:** `Counter` là `dict` đặc biệt, key chưa có trả `0` → không cần lo chỉ số `-v` vượt `hi`.

## 19. Tổng kết — template + checklist nộp bài

**Template nhớ nhanh (tham lam hai pha):**

```python
class Solution:
    def largestSumAfterKNegations(self, nums: list[int], k: int) -> int:
        nums.sort()
        for i in range(len(nums)):
            if nums[i] >= 0 or k == 0:
                break
            nums[i] = -nums[i]
            k -= 1
        total = sum(nums)
        if k % 2 == 1:
            total -= 2 * min(nums)
        return total
```

**Câu thần chú:** *"Âm to đổi trước — lượt thừa xem chẵn lẻ — lẻ thì trừ hai lần số nhỏ nhất."*

**Checklist trước khi submit:**
- [ ] Đã sắp xếp (hoặc heap) trước khi đổi số âm?
- [ ] Dừng đổi khi `k == 0`?
- [ ] Xử lý lượt thừa: `k % 2 == 1`?
- [ ] Lấy `min` **sau** pha 1 (không phải trên mảng gốc)?
- [ ] Trừ `2 * min`, không phải `min`?
- [ ] Test với ca có số `0`, ca toàn dương, ca `k` nhỏ hơn số số âm?